<a href="https://colab.research.google.com/github/tezzk/houston-energy-burden/blob/main/energy_burden.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Houston Energy Burden Analysis (2022–2024)

**Question:** How did energy burden change for Harris County households from 2022 to 2024, and who carries it most?

**Data:** U.S. Census ACS 1-Year PUMS (household files), Harris County (39 PUMAs)

**Energy burden** = yearly electricity + gas cost ÷ household income

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd

In [ ]:
cols = ["PUMA", "HINCP", "ADJINC", "ELEP", "ELEFP", "GASP", "GASFP", "TEN", "WGTP"]


In [ ]:
harris = [str(n).zfill(5) for n in range(4601, 4641) if n != 4632]



In [ ]:
# Weighted median: sort by burden, find the point where half the weight is below
def wmedian(values, weights):
    d = pd.DataFrame({"v": values, "w": weights}).sort_values("v")
    half = d["w"].sum() / 2
    return d.loc[d["w"].cumsum() >= half, "v"].iloc[0]
# Summary for any group: households, weighted median, % high or severe
def summary(g):
    w = g["WGTP"]
    return pd.Series({
        "households": int(w.sum()),
        "median_burden_%": round(wmedian(g["burden"], w), 2),
        "pct_high_or_severe": round(w[g["burden"] >= 6].sum() / w.sum() * 100, 1)
    })



In [ ]:
def load_clean(year):
    # 1. Load only our 9 columns, PUMA as text
    path = f"/content/drive/MyDrive/energy_burden/psam_h48_{year}.csv"
    df = pd.read_csv(path, usecols=cols, dtype={"PUMA": str})

    # 2. Harris County only
    df = df[df["PUMA"].isin(harris)].copy()
    print(year, "Harris County rows:", len(df))

    # 3. Cleaning (same steps as before)
    df = df[df["TEN"].notna()]
    df = df[df["HINCP"] > 0]
    df = df[df["ELEFP"] == 3]
    df = df[df["GASFP"].isin([2, 3, 4])]
    print(year, "after cleaning:", len(df))

    # 4. Calculations
    df["gas"] = df["GASP"].where(df["GASFP"] == 4, 0)
    df["income"] = df["HINCP"] * df["ADJINC"] / 1_000_000
    df["energy_yr"] = (df["ELEP"] + df["gas"]) * 12
    df["burden"] = df["energy_yr"] / df["income"] * 100

    # 5. Group labels
    df["tenure"] = df["TEN"].map({1: "Owner", 2: "Owner", 3: "Renter", 4: "No rent paid"})
    df["income_group"] = pd.cut(df["income"], bins=[0, 30000, 60000, 100000, float("inf")],
                                labels=["Under $30k", "$30k-60k", "$60k-100k", "$100k+"], right=False)
    df["year"] = year
    return df

c23 = load_clean(2023)
c24 = load_clean(2024)
all_years = pd.concat([c23, c24])

2023 Harris County rows: 16974
2023 after cleaning: 13747
2024 Harris County rows: 16624
2024 after cleaning: 13205


In [ ]:
print("=== Overall ===")
print(all_years.groupby("year").apply(summary, include_groups=False))

print("\n=== By tenure ===")
ow_rent = all_years[all_years["tenure"].isin(["Owner", "Renter"])]
print(ow_rent.groupby(["tenure", "year"]).apply(summary, include_groups=False))

print("\n=== By income ===")
print(all_years.groupby(["income_group", "year"], observed=True).apply(summary, include_groups=False))

=== Overall ===
      households  median_burden_%  pct_high_or_severe
year                                                 
2023   1631441.0             3.30                27.1
2024   1633758.0             3.08                25.9

=== By tenure ===
             households  median_burden_%  pct_high_or_severe
tenure year                                                 
Owner  2023    961110.0             3.10                24.1
       2024    938976.0             2.85                22.3
Renter 2023    658123.0             3.65                31.0
       2024    677739.0             3.50                30.5

=== By income ===
                   households  median_burden_%  pct_high_or_severe
income_group year                                                 
Under $30k   2023    271794.0            13.08                85.2
             2024    274113.0            13.29                84.7
$30k-60k     2023    363264.0             5.10                39.1
             2024    340780.0

In [ ]:
for y, d in all_years.groupby("year"):
    print(y, "median income:", round(wmedian(d["income"], d["WGTP"])),
          "| median yearly energy cost:", round(wmedian(d["energy_yr"], d["WGTP"])))

2023 median income: 76464 | median yearly energy cost: 2640
2024 median income: 78783 | median yearly energy cost: 2520


In [ ]:
   test = pd.read_csv("/content/drive/MyDrive/energy_burden/psam_h48_2022.csv",
                      usecols=["SERIALNO"], nrows=3)
   print(test)

        SERIALNO
0  2022GQ0000010
1  2022GQ0000023
2  2022GQ0000024


In [ ]:
 def load_clean(year):
    # 1. Load only our 9 columns, PUMA as text
    path = f"/content/drive/MyDrive/energy_burden/psam_h48_{year}.csv"
    df = pd.read_csv(path, usecols=cols, dtype={"PUMA": str})

    # 2. Harris County only
    df = df[df["PUMA"].isin(harris)].copy()
    print(year, "Harris County rows:", len(df))

    # 3. Cleaning (same steps as before)
    df = df[df["TEN"].notna()]
    df = df[df["HINCP"] > 0]
    df = df[df["ELEFP"] == 3]
    df = df[df["GASFP"].isin([2, 3, 4])]
    print(year, "after cleaning:", len(df))

    # 4. Calculations
    df["gas"] = df["GASP"].where(df["GASFP"] == 4, 0)
    df["income"] = df["HINCP"] * df["ADJINC"] / 1_000_000
    df["energy_yr"] = (df["ELEP"] + df["gas"]) * 12
    df["burden"] = df["energy_yr"] / df["income"] * 100

    # 5. Group labels
    df["tenure"] = df["TEN"].map({1: "Owner", 2: "Owner", 3: "Renter", 4: "No rent paid"})
    df["income_group"] = pd.cut(df["income"], bins=[0, 30000, 60000, 100000, float("inf")],
                                labels=["Under $30k", "$30k-60k", "$60k-100k", "$100k+"], right=False)
    df["year"] = year
    return df

c22 = load_clean(2022)
c23 = load_clean(2023)
c24 = load_clean(2024)
all_years = pd.concat([c22, c23, c24])

2022 Harris County rows: 16764
2022 after cleaning: 13565
2023 Harris County rows: 16974
2023 after cleaning: 13747
2024 Harris County rows: 16624
2024 after cleaning: 13205


In [ ]:
print("=== Overall ===")
print(all_years.groupby("year").apply(summary, include_groups=False))

print("\n=== By tenure ===")
ow_rent = all_years[all_years["tenure"].isin(["Owner", "Renter"])]
print(ow_rent.groupby(["tenure", "year"]).apply(summary, include_groups=False))

print("\n=== By income ===")
print(all_years.groupby(["income_group", "year"], observed=True).apply(summary, include_groups=False))

=== Overall ===
      households  median_burden_%  pct_high_or_severe
year                                                 
2022   1640582.0             3.14                24.1
2023   1631441.0             3.30                27.1
2024   1633758.0             3.08                25.9

=== By tenure ===
             households  median_burden_%  pct_high_or_severe
tenure year                                                 
Owner  2022    933201.0             2.93                22.3
       2023    961110.0             3.10                24.1
       2024    938976.0             2.85                22.3
Renter 2022    689668.0             3.45                26.0
       2023    658123.0             3.65                31.0
       2024    677739.0             3.50                30.5

=== By income ===
                   households  median_burden_%  pct_high_or_severe
income_group year                                                 
Under $30k   2022    288885.0            11.51        

In [ ]:
for y, d in all_years.groupby("year"):
    print(y, "median income:", round(wmedian(d["income"], d["WGTP"])),
          "| median yearly energy cost:", round(wmedian(d["energy_yr"], d["WGTP"])))

2022 median income: 71919 | median yearly energy cost: 2400
2023 median income: 76464 | median yearly energy cost: 2640
2024 median income: 78783 | median yearly energy cost: 2520


In [ ]:
folder = "/content/drive/MyDrive/energy_burden/"

overall = all_years.groupby("year").apply(summary, include_groups=False).reset_index()
by_tenure = ow_rent.groupby(["tenure", "year"]).apply(summary, include_groups=False).reset_index()
by_income = all_years.groupby(["income_group", "year"], observed=True).apply(summary, include_groups=False).reset_index()

overall.to_csv(folder + "result_overall.csv", index=False)
by_tenure.to_csv(folder + "result_by_tenure.csv", index=False)
by_income.to_csv(folder + "result_by_income.csv", index=False)
print("Saved")

Saved


In [ ]:
by_income_tenure = ow_rent.groupby(["income_group", "tenure", "year"], observed=True).apply(summary, include_groups=False).reset_index()
by_income_tenure.to_csv(folder + "result_by_income_tenure.csv", index=False)


## Key Findings

- **Overall:** About 1 in 4 Harris County households had a high or severe energy burden in 2024.
- **2023 spike:** Energy bills rose about 10% while income rose about 6%, so burden went up.
- **Who recovered:** By 2024, Owners returned to 2022 levels, but renters and Under -30k did not.
- **Surprise:** Renters look worse overall, but at the same income, Owners actually have higher burden.
- **Most at risk:** Households earning under $30k: about 84.7% had high or severe burden.

## Limits
- Survey data (a sample), so small changes of about 1 point may be random noise.
- Households whose utilities are included in rent were removed, because their real energy cost is unknown.
- Dollar amounts are not adjusted for inflation (burden is a ratio, so it isn't affected).
- The data shows what changed, not why. The cause of the 2023 spike isn't proven.
- Area is Harris County as a whole, not individual neighborhoods.